# Silver Layer Data Validation
This notebook runs data integrity tests on the `silver`  tables to ensure the Medallion Pipeline correctly transformed the raw files.
> Note: The raw data and `.db` file are excluded from source control.

In [1]:
import duckdb
import pandas as pd

conn = duckdb.connect('../lakehouse_db/aki_lakehouse.db', read_only=True)
print('Connected to Lakehouse Database successfully.')

Connected to Lakehouse Database successfully.


## Base Data

### Test 1: Materialization Verification
Ensure that all silver tables have been populated with records.

In [2]:
tables = [
    'silver_stays', 'silver_creatinine', 'silver_urine_output', 
    'silver_aki_baselines', 'silver_kdigo_labels', 'silver_vitals', 'silver_comorbidities'
]

for table in tables:
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table}: {count:,} rows')

silver_stays: 73,181 rows
silver_creatinine: 599,781 rows
silver_urine_output: 3,247,199 rows
silver_aki_baselines: 72,951 rows
silver_kdigo_labels: 3,830,747 rows
silver_vitals: 11,199,738 rows
silver_comorbidities: 180,640 rows


### Test 2: Stay & Patient ID Integrity
Verify that the core identifiers extracted from FHIR references are not null.

In [3]:
integrity_check = conn.execute('''
    SELECT 
        COUNT(*) as total_stays,
        SUM(CASE WHEN patient_id IS NULL THEN 1 ELSE 0 END) as null_patients,
        SUM(CASE WHEN stay_id IS NULL THEN 1 ELSE 0 END) as null_stays
    FROM silver_stays
''').df()
display(integrity_check)

,total_stays,null_patients,null_stays
0,73181,0.0,0.0


# KDIGO Labels

### Test 3: KDIGO Label Validation (Creatinine Spike)
Sample records where an Acute Kidney Injury was flagged due to a 48-hour absolute creatinine spike.

In [4]:
scr_spike_sample = conn.execute('''
    SELECT stay_id, charttime, current_scr, baseline_creatinine, criteria_absolute_spike, is_aki
    FROM silver_kdigo_labels
    WHERE criteria_absolute_spike = TRUE 
    LIMIT 5
''').df()
display(scr_spike_sample)

,stay_id,charttime,current_scr,baseline_creatinine,criteria_absolute_spike,is_aki
0,b3527709-38f1-52f0-8b7f-cc54a73d6199,2186-06-19 22:00:00,3.2,2.4,True,True
1,b3527709-38f1-52f0-8b7f-cc54a73d6199,2186-06-20 17:00:00,4.2,2.4,True,True
2,b69724a9-4a41-5bd6-a323-1754af5c29c3,2138-07-03 01:00:00,3.8,2.2,True,True
3,b8683411-d799-5a1e-9a1c-8cd09c941be0,2146-02-15 05:00:00,4.7,1.6,True,True
4,b9a3c461-8595-5aae-915b-38e1cb20e142,2148-06-04 04:00:00,0.8,0.6,True,True


### Test 4: KDIGO Label Validation (7-Day Relative Spike)
Sample records where Acute Kidney Injury was flagged due to a creatinine increase of ≥1.5x the minimum value within the prior 7 days.

In [5]:
scr_relative_sample = conn.execute('''
    SELECT stay_id, charttime, current_scr, baseline_creatinine, criteria_relative_rolling, is_aki
    FROM silver_kdigo_labels
    WHERE criteria_relative_rolling = TRUE 
    LIMIT 5
''').df()
display(scr_relative_sample)


,stay_id,charttime,current_scr,baseline_creatinine,criteria_relative_rolling,is_aki
0,b3527709-38f1-52f0-8b7f-cc54a73d6199,2186-06-20 17:00:00,4.2,2.4,True,True
1,b4178359-0049-57a6-a883-2aedbfba26d0,2188-10-31 02:00:00,0.2,0.2,True,True
2,b69724a9-4a41-5bd6-a323-1754af5c29c3,2138-07-03 01:00:00,3.8,2.2,True,True
3,b70f6323-2503-5b34-be28-6da1ba3d922b,2140-08-19 04:00:00,1.0,0.9,True,True
4,b8683411-d799-5a1e-9a1c-8cd09c941be0,2146-02-15 05:00:00,4.7,1.6,True,True


### Test 5: KDIGO Label Validation (Urine Output Drop)
Sample records where an Acute Kidney Injury was flagged due to low urine output over a 6-hour window.

In [6]:
uo_drop_sample = conn.execute('''
    SELECT stay_id, charttime, criteria_urine_output, is_aki
    FROM silver_kdigo_labels
    WHERE criteria_urine_output = TRUE 
    LIMIT 5
''').df()
display(uo_drop_sample)

,stay_id,charttime,criteria_urine_output,is_aki
0,b2ea5e4e-63f1-5666-81bb-85837adfa093,2150-05-01 03:00:00,True,True
1,b2ea5e4e-63f1-5666-81bb-85837adfa093,2150-05-01 04:00:00,True,True
2,b2ea5e4e-63f1-5666-81bb-85837adfa093,2150-05-01 05:00:00,True,True
3,b2ea5e4e-63f1-5666-81bb-85837adfa093,2150-05-01 06:00:00,True,True
4,b2ea5e4e-63f1-5666-81bb-85837adfa093,2150-05-01 07:00:00,True,True


## Supplementary data

### Test 6: Comorbidities Data Extraction
Verify that boolean flags for chronic conditions were properly extracted from the ICD code structures.

In [7]:
comorbidities_sample = conn.execute('''
    SELECT 
        patient_id,
        history_ckd,
        history_diabetes,
        history_chf,
        history_hypertension
    FROM silver_comorbidities
    WHERE history_ckd = 1 OR history_diabetes = 1
    LIMIT 5
''').df()
display(comorbidities_sample)

,patient_id,history_ckd,history_diabetes,history_chf,history_hypertension
0,06eeac8a-6922-59e3-8aa2-cf2829377def,1,0,0,0
1,072d0578-b27e-5b05-a57e-6681a30cd83f,1,1,1,0
2,072e5967-bf71-5c89-92eb-769de4c0a219,0,1,1,0
3,07507650-4cae-5143-80b2-de19c6543c32,1,1,0,0
4,078bb0d2-0db4-5fac-a344-4220583dac18,0,1,0,0


### Test 7: Medication Administration Validation
Sample records from the `silver_medications` table, ensuring both DateTime pushes and Period infusions were captured properly.

In [8]:
med_sample = conn.execute('''
    SELECT stay_id, charttime, endtime, medication_name, dose, dose_unit
    FROM silver_medications
    WHERE medication_name IS NOT NULL
    LIMIT 5
''').df()
display(med_sample)

,stay_id,charttime,endtime,medication_name,dose,dose_unit
0,4cb98b80-af38-537a-b354-9b9c87678991,2131-01-13 03:19:00,2131-01-13 04:21:00,Norepinephrine,0.330131,mg
1,4cb98b80-af38-537a-b354-9b9c87678991,2131-01-13 03:19:00,2131-01-13 04:21:00,NaCl 0.9%,10.316604,ml
2,4cb98b80-af38-537a-b354-9b9c87678991,2131-01-13 04:21:00,2131-01-13 05:39:00,Norepinephrine,0.276972,mg
3,4cb98b80-af38-537a-b354-9b9c87678991,2131-01-13 04:21:00,2131-01-13 05:39:00,NaCl 0.9%,8.655381,ml
4,4cb98b80-af38-537a-b354-9b9c87678991,2131-01-13 05:36:00,2131-01-13 07:36:00,NaCl 0.9%,99.999998,ml


## Final check

### Test 8: Sense Check for Final AKI Cohort Distribution
Count the total number of unique ICU stays that developed AKI vs those that did not.

In [ ]:
cohort_dist = conn.execute('''
    WITH stay_level AS (
        SELECT stay_id, MAX(CAST(is_aki AS INT)) as developed_aki
        FROM silver_kdigo_labels
        GROUP BY stay_id
    )
    SELECT developed_aki, COUNT(*) as number_of_stays
    FROM stay_level
    GROUP BY developed_aki
''').df()
display(cohort_dist)

,developed_aki,number_of_stays
0,0,31192
1,1,41938


: 

~42.6% developed AKI. Clinical data shows that between 40% and 55% of all ICU patients develop some stage of AKI during their stay, so this looks just about right.